# PhotoWalk — Colab Training Notebook

**One-click 3D Gaussian Splatting from phone images**

---

## Before running

1. **Enable GPU:** `Runtime → Change runtime type → T4 GPU`
2. **Choose mode** in the Configuration cell below:
   - `"test_dataset"` — downloads the gerrard-hall dataset automatically, no upload needed
   - `"own_scene"` — uses your phone photos uploaded to Google Drive
3. **Click `Runtime → Run all`**

**Estimated time:** ~45–60 min total (install: 15–20 min, train: 20–30 min)

---

**Output:** a `.ply` file downloaded to your machine, ready for SuperSplat.

## ⚙️ Configuration — Edit this cell only

Set `MODE` and (if using own scene) update the Drive path.

In [1]:
from pathlib import Path

MODE = "test_dataset"           # Using test dataset (no photos needed)
SCENE_NAME = "gerrard-hall"

MAX_ITERATIONS = 10000          # Start with 10000 for faster testing (change to 30000 later)

WORK_DIR = Path("/content/photowalk")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode        : {MODE}")
print(f"Scene       : {SCENE_NAME}")
print(f"Iterations  : {MAX_ITERATIONS}")

Mode        : test_dataset
Scene       : gerrard-hall
Iterations  : 10000


## 0. GPU Check

In [2]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {props.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU"
    )

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


## 1. Install Nerfstudio

Installs PyTorch (CUDA build) → tiny-cuda-nn → gsplat → Nerfstudio.

**Takes ~15–20 minutes. Do not interrupt.**

In [3]:
import subprocess, sys

def run(cmd, label, check=True):
    print(f"\n>>> {label}")
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{label} failed (exit {r.returncode})")
    return r

# Detect CUDA version
nvcc = subprocess.run("nvcc --version", shell=True, capture_output=True, text=True).stdout
torch_build = "cu121" if "12." in nvcc else "cu118"
torch_index  = f"https://download.pytorch.org/whl/{torch_build}"
print(f"CUDA build target: {torch_build}")
print(nvcc[:200])

CUDA build target: cu121
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.355


In [4]:
# PyTorch with CUDA — skip if already correct
import torch
need_torch = not torch.cuda.is_available() or "+cu" not in torch.__version__

if need_torch:
    run(
        f"pip install -q torch==2.1.2+{torch_build} torchvision==0.16.2+{torch_build} "
        f"--extra-index-url {torch_index}",
        "Installing PyTorch+CUDA"
    )
else:
    print(f"PyTorch {torch.__version__} already has CUDA — skipping.")

PyTorch 2.10.0+cu128 already has CUDA — skipping.


In [5]:
# tiny-cuda-nn: required by Nerfstudio, compiles from source (~10 min)
# If this fails, see Troubleshooting at the bottom of the notebook.
run("pip install -q ninja", "Installing ninja")
run(
    "pip install -q git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch",
    "Installing tiny-cuda-nn (compiles from source — takes ~10 min)"
)


>>> Installing ninja

>>> Installing tiny-cuda-nn (compiles from source — takes ~10 min)


CompletedProcess(args='pip install -q git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch', returncode=0)

In [6]:
run("pip install -q nerfstudio", "Installing Nerfstudio")

# Verify CLIs are available
for cli in ["ns-train", "ns-process-data", "ns-export"]:
    r = run(f"{cli} --help > /dev/null 2>&1", f"Verifying {cli}", check=False)
    status = "OK" if r.returncode == 0 else "MISSING"
    print(f"  {cli}: {status}")

print("\nNerfstudio installation complete.")


>>> Installing Nerfstudio

>>> Verifying ns-train
  ns-train: OK

>>> Verifying ns-process-data
  ns-process-data: OK

>>> Verifying ns-export
  ns-export: OK

Nerfstudio installation complete.


## 2. Get Image Data

In [7]:
from pathlib import Path
import zipfile
import subprocess

WORK_DIR = Path("/content/photowalk")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading and setting up gerrard-hall dataset...")

zip_path = WORK_DIR / "gerrard-hall.zip"
extract_path = WORK_DIR / "gerrard-hall"

if not (extract_path / "images").exists():
    if not zip_path.exists():
        print("Downloading...")
        subprocess.run([
            "wget", "-q", "--show-progress",
            "-O", str(zip_path),
            "https://github.com/colmap/colmap/releases/download/3.11.1/gerrard-hall.zip"
        ], check=True)

    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(WORK_DIR)
    print("✅ Extraction complete!")
else:
    print("✅ Dataset already extracted.")

# Set global paths
IMAGE_DIR = extract_path / "images"
NS_DATA_DIR = WORK_DIR / "ns_data"
NS_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nImages found: {len(list(IMAGE_DIR.glob('*')))}")
print(f"IMAGE_DIR: {IMAGE_DIR}")

Downloading...
Extracting...
✅ Extraction complete!

Images found: 100
IMAGE_DIR: /content/photowalk/gerrard-hall/images


## 3. Validate Images

Checks blur, brightness, and resolution. Prints a summary and flags problems.

In [11]:
import cv2
import numpy as np
from pathlib import Path

# Use the correct path from the download cell
IMAGE_DIR = WORK_DIR / "gerrard-hall" / "images"

images = sorted(IMAGE_DIR.glob("*.jpg")) + sorted(IMAGE_DIR.glob("*.png")) + \
         sorted(IMAGE_DIR.glob("*.JPG")) + sorted(IMAGE_DIR.glob("*.PNG"))

print(f"✅ Found {len(images)} images in: {IMAGE_DIR}")

if len(images) > 0:
    print("Sample:", [p.name for p in images[:5]])

    # Quick quality check on first 30 images
    blur_scores = []
    for p in images[:30]:
        img = cv2.imread(str(p))
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.Laplacian(gray, cv2.CV_64F).var()
        blur_scores.append(blur)

    print(f"Average blur score: {np.mean(blur_scores):.1f} (higher = sharper)")
else:
    print("❌ No images found. Check the path above.")

✅ Found 100 images in: /content/photowalk/gerrard-hall/images
Sample: ['IMG_2331.JPG', 'IMG_2332.JPG', 'IMG_2333.JPG', 'IMG_2334.JPG', 'IMG_2335.JPG']
Average blur score: 150.7 (higher = sharper)


## 4. Preprocess Images

Renames images to `frame_XXXX.jpg` and optionally resizes to a max dimension.

Skipped for gerrard-hall (images already named consistently by COLMAP).

In [9]:
print("Using raw images directly (no preprocessing needed for own_scene in this setup)")
IMAGE_DIR = WORK_DIR / "gerrard-hall" / "images"

Using raw images directly (no preprocessing needed for own_scene in this setup)


## 5. COLMAP — Camera Pose Estimation

Estimates where each camera was when each photo was taken.

- **gerrard-hall**: uses pre-computed COLMAP sparse reconstruction — fast (~1 min)
- **own scene**: runs full COLMAP feature extraction + matching + bundle adjustment (~10–30 min)

In [14]:
import subprocess
import os

print("Installing COLMAP via apt...")
subprocess.run("apt-get install -qq colmap", shell=True, check=True)

os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

result = subprocess.run("which colmap", shell=True, capture_output=True, text=True)
print(f"COLMAP found at: {result.stdout.strip()}")

result = subprocess.run("colmap --version", shell=True, capture_output=True, text=True)
print(f"Version: {result.stdout.strip()}")

Installing COLMAP via apt...
COLMAP found at: /usr/bin/colmap
Version: 


In [15]:
import subprocess
from pathlib import Path

WORK_DIR = Path("/content/photowalk")
IMAGE_DIR = WORK_DIR / "gerrard-hall" / "images"
SPARSE_DIR = WORK_DIR / "gerrard-hall" / "sparse"
SPARSE_BIN_DIR = SPARSE_DIR / "0"
NS_DATA_DIR = WORK_DIR / "ns_data"

SPARSE_BIN_DIR.mkdir(exist_ok=True)
NS_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Convert txt -> bin
print("Converting COLMAP txt -> bin...")
convert_cmd = (
    f"colmap model_converter "
    f"--input_path '{SPARSE_DIR}' "
    f"--output_path '{SPARSE_BIN_DIR}' "
    f"--output_type BIN"
)
result = subprocess.run(convert_cmd, shell=True, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

# Step 2: Run ns-process-data with pre-computed sparse
print("Running ns-process-data with pre-computed sparse...")
cmd = (
    f"ns-process-data images "
    f"--data '{IMAGE_DIR}' "
    f"--output-dir '{NS_DATA_DIR}' "
    f"--skip-colmap "
    f"--colmap-model-path '{SPARSE_BIN_DIR}'"
)
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
print(f"Exit code: {result.returncode}")

if (NS_DATA_DIR / "transforms.json").exists():
    print("✅ Success! transforms.json created.")
else:
    print("Still failed.")

Converting COLMAP txt -> bin...


Running ns-process-data with pre-computed sparse...
[03:49:41] 🎉 Done copying images with prefix 'frame_'.                                        process_data_utils.py:348
[03:49:43] 🎉 🎉 🎉 All DONE 🎉 🎉 🎉                                                images_to_nerfstudio_dataset.py:135
           Starting with 100 images                                                  images_to_nerfstudio_dataset.py:138
           Colmap matched 100 images                                                 images_to_nerfstudio_dataset.py:138
           COLMAP found poses for all images, CONGRATS!                              images_to_nerfstudio_dataset.py:138


Exit code: 0
✅ Success! transforms.json created.


## 6. Train Splatfacto (3D Gaussian Splatting)

Optimizes 3D Gaussians from the camera poses + images.

- T4 GPU, 30k iterations: ~20–25 min
- T4 GPU, 10k iterations: ~7–10 min (quick test, lower quality)

In [20]:
from pathlib import Path
import subprocess

OUTPUT_DIR = WORK_DIR / "outputs" / SCENE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NS_DATA_DIR = WORK_DIR / "ns_data"

train_cmd = (
    f"ns-train splatfacto "
    f"--data '{NS_DATA_DIR}' "
    f"--output-dir '{OUTPUT_DIR}' "
    f"--max-num-iterations {MAX_ITERATIONS} "
    f"--viewer.quit-on-train-completion True"
)

print(f"Starting Splatfacto training ({MAX_ITERATIONS} iterations)...")
print(f"Scene: {SCENE_NAME}")
print(f"Data path: {NS_DATA_DIR}")
print(f"Output path: {OUTPUT_DIR}")
print(f"Command: {train_cmd}")
print("\n" + "="*80)

result = subprocess.run(train_cmd, shell=True)

if result.returncode != 0:
    raise RuntimeError(f"ns-train failed (exit {result.returncode})")

print("\n🎉 Training completed successfully!")

configs = sorted(OUTPUT_DIR.glob("**/config.yml"), key=lambda p: p.stat().st_mtime)
if not configs:
    raise FileNotFoundError(f"No config.yml found under {OUTPUT_DIR}")

CONFIG_PATH = configs[-1]
print(f"✅ Config ready: {CONFIG_PATH}")

Starting Splatfacto training (10000 iterations)...
Scene: gerrard-hall
Data path: /content/photowalk/ns_data
Output path: /content/photowalk/outputs/gerrard-hall
Command: ns-train splatfacto --data '/content/photowalk/ns_data' --output-dir '/content/photowalk/outputs/gerrard-hall' --max-num-iterations 10000 --viewer.quit-on-train-completion True


🎉 Training completed successfully!
✅ Config ready: /content/photowalk/outputs/gerrard-hall/ns_data/splatfacto/2026-05-06_035417/config.yml


## 7. Export Gaussian Splat (.ply)

Exports the trained model as a `.ply` file compatible with SuperSplat.

In [23]:
from pathlib import Path
import subprocess

EXPORT_DIR = WORK_DIR / "exports" / SCENE_NAME
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Patch torch.load to use weights_only=False, then run export in same process
export_script = f"""
import torch
_original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_load(*args, **kwargs)
torch.load = patched_load

import sys
sys.argv = [
    'ns-export', 'gaussian-splat',
    '--load-config', '{CONFIG_PATH}',
    '--output-dir', '{EXPORT_DIR}'
]

from nerfstudio.scripts.exporter import entrypoint
entrypoint()
"""

result = subprocess.run(f'python3 -c "{export_script}"', shell=True, capture_output=True, text=True)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
print(f"Exit code: {result.returncode}")

ply_files = list(EXPORT_DIR.glob("*.ply"))
if ply_files:
    print(f"✅ Export successful: {ply_files[0]}")
else:
    print("Still failed.")

Loading latest checkpoint from load_dir
✅ Done loading checkpoint from 
/content/photowalk/outputs/gerrard-hall/ns_data/splatfacto/2026-05-06_035417/nerfstudio_models/step-000009999.ckpt
0 Gaussians have NaN/Inf and 4 have low opacity, only export 296526/296530

2026-05-06 04:11:06.792377: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecat

## 8. Download .ply

Downloads the `.ply` to your local machine.

In [25]:
from google.colab import files
from pathlib import Path

EXPORT_DIR = WORK_DIR / "exports" / SCENE_NAME
ply_files = list(EXPORT_DIR.glob("*.ply"))

if not ply_files:
    raise FileNotFoundError(f"No .ply file found in {EXPORT_DIR}")

PLY_PATH = ply_files[0]
size_mb = PLY_PATH.stat().st_size / 1e6

print(f"Downloading {PLY_PATH.name} ({size_mb:.1f} MB)...")
files.download(str(PLY_PATH))

print()
print("Next steps:")
print("  1. Go to https://playcanvas.com/supersplat/editor")
print("  2. Drag-and-drop the downloaded .ply file")
print("  3. Click Publish → copy the viewer URL")
print(f"  4. Paste URL into web/src/data/scenes.js under '{SCENE_NAME}'")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Record Experiment Result

Fill in the fields below after visually inspecting the splat in SuperSplat,
then copy the output JSON into `experiments.json` locally.

In [ ]:
# ── Fill in these fields after visual inspection ──────────────────────────────
experiment = {
    "scene_name":              SCENE_NAME,
    "num_images":              len(images),
    "capture_condition":       "arc path, 60% overlap",   # describe how photos were taken
    "lighting":                "bright outdoor",           # bright / dim / mixed
    "texture_level":           "high",                     # high / medium / very low
    "reconstruction_success":  "yes",                      # yes / partial / no
    "major_artifacts":         "none",                     # none / floaters / missing geometry
    "viewer_file_size_mb":     round(size_mb, 1),
    "qualitative_score":       4,                          # 1 (failed) – 5 (excellent)
    "notes":                   "",
}

print(json.dumps(experiment, indent=2))
print("\nCopy this into experiments.json on your local machine.")

---

## 10. Resume after Session Timeout

If Colab timed out after training but before export, run this cell to re-export without re-training.

> **Note:** Colab's `/content/` does not persist across sessions.
> If you lost the training outputs, you must re-train.
> To avoid this, save outputs to Drive: `!cp -r /content/photowalk/outputs /content/drive/MyDrive/photowalk/outputs`

In [ ]:
# Run this cell only if you need to resume a previous session.
from pathlib import Path
import subprocess

WORK_DIR   = Path("/content/photowalk")
OUTPUT_DIR = WORK_DIR / "outputs"
EXPORT_DIR = WORK_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

configs = sorted(OUTPUT_DIR.glob("**/config.yml"), key=lambda p: p.stat().st_mtime)
if not configs:
    raise FileNotFoundError(
        f"No config.yml under {OUTPUT_DIR}\n"
        "The training outputs were lost. Re-run the training cell."
    )

CONFIG_PATH = configs[-1]
print(f"Found config: {CONFIG_PATH}")

result = subprocess.run(
    f"ns-export gaussian-splat --load-config '{CONFIG_PATH}' --output-dir '{EXPORT_DIR}'",
    shell=True
)
if result.returncode != 0:
    raise RuntimeError("Export failed — check output above.")

ply_files = list(EXPORT_DIR.glob("*.ply"))
if ply_files:
    from google.colab import files
    files.download(str(ply_files[0]))
    print(f"Downloaded: {ply_files[0].name}")

---

## Troubleshooting

| Problem | Fix |
|---|---|
| `No GPU detected` | Runtime → Change runtime type → T4 GPU |
| `tiny-cuda-nn fails to install` | Try: `pip install tinycudann --find-links https://github.com/NVlabs/tiny-cuda-nn/releases` |
| `ns-train not found` | Restart the runtime (`Runtime → Restart`), then re-run from the install cell |
| `--skip-colmap not recognized` | Nerfstudio version mismatch — remove the flag in the COLMAP cell (COLMAP will re-run, ~20 min) |
| `CUDA out of memory` | Add `--pipeline.model.num-random-init-gaussians 10000` to the `ns-train` command |
| `transforms.json has < 10 frames` | Scene has low texture or bad overlap — check validation output |
| `No .ply file after export` | Training didn't finish — reduce `MAX_ITERATIONS` and re-train |
| Session timed out | Use the Resume cell (Section 10) — but only if `/content/` outputs still exist |

---

## After SuperSplat

1. Go to https://playcanvas.com/supersplat/editor
2. Drag-and-drop your `.ply` file into the editor
3. Click **Publish** → copy the shareable URL
4. Open `web/src/data/scenes.js` locally and update the `viewer_url` field for your scene
5. Run `cd web && npm run dev` to verify the landing page shows the viewer